In [ ]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [ ]:
!pip install -q torch transformers pillow pandas


In [ ]:
import os

os.makedirs("vqa_mini/images", exist_ok=True)
print("Folders created")


Folders created


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving v2_mscoco_val2014_annotations.json to v2_mscoco_val2014_annotations.json


In [ ]:
import json, random

N = 20  # number of samples (change to 10 if needed)

with open("vqa_mini/v2_OpenEnded_mscoco_val2014_questions.json") as f:
    questions = json.load(f)["questions"]

with open("vqa_mini/v2_mscoco_val2014_annotations.json") as f:
    annotations = json.load(f)["annotations"]

ann_by_qid = {a["question_id"]: a for a in annotations}

paired = [(q, ann_by_qid[q["question_id"]])
          for q in questions if q["question_id"] in ann_by_qid]

random.seed(42)
subset = random.sample(paired, N)

mini_data = []
for q, a in subset:
    mini_data.append({
        "question_id": q["question_id"],
        "image_id": q["image_id"],
        "question": q["question"],
        "answers": [x["answer"] for x in a["answers"]]
    })

with open("vqa_subset.json", "w") as f:
    json.dump(mini_data, f, indent=2)

print(f"Created mini dataset with {N} samples")


Created mini dataset with 20 samples


In [ ]:
import urllib.request

BASE_URL = "http://images.cocodataset.org/val2014/"
with open("vqa_subset.json") as f:
    data = json.load(f)

image_ids = sorted({x["image_id"] for x in data})

print(f"Downloading {len(image_ids)} images")

for img_id in image_ids:
    fname = f"COCO_val2014_{img_id:012d}.jpg"
    url = BASE_URL + fname
    out = f"vqa_mini/images/{fname}"

    if not os.path.exists(out):
        urllib.request.urlretrieve(url, out)
        print("Downloaded:", fname)

print("Image download complete")


Downloaded: COCO_val2014_000000004396.jpg
Downloaded: COCO_val2014_000000032364.jpg
Downloaded: COCO_val2014_000000100187.jpg
Downloaded: COCO_val2014_000000117869.jpg
Downloaded: COCO_val2014_000000134483.jpg
Downloaded: COCO_val2014_000000143450.jpg
Downloaded: COCO_val2014_000000145824.jpg
Downloaded: COCO_val2014_000000223174.jpg
Downloaded: COCO_val2014_000000248069.jpg
Downloaded: COCO_val2014_000000248767.jpg
Downloaded: COCO_val2014_000000278161.jpg
Downloaded: COCO_val2014_000000297944.jpg
Downloaded: COCO_val2014_000000302260.jpg
Downloaded: COCO_val2014_000000346214.jpg
Downloaded: COCO_val2014_000000376667.jpg
Downloaded: COCO_val2014_000000397303.jpg
Downloaded: COCO_val2014_000000406647.jpg
Downloaded: COCO_val2014_000000413522.jpg
Downloaded: COCO_val2014_000000455506.jpg
Downloaded: COCO_val2014_000000499727.jpg
Image download complete


It looks like one of the JSON files, specifically `v2_OpenEnded_mscoco_val2014_questions.json`, is corrupted or was not fully uploaded, leading to the `JSONDecodeError`. Please re-upload this file. After re-uploading, re-run the cell above this one (cell `znxKgEgzFk3b`) and then proceed with the rest of the notebook.

In [ ]:
from transformers import pipeline
from PIL import Image

vqa = pipeline("visual-question-answering",
               model="Salesforce/blip-vqa-base")

print("VQA model loaded")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.54G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForQuestionAnswering LOAD REPORT from: Salesforce/blip-vqa-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 
text_encoder.embeddings.position_ids      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


VQA model loaded


In [ ]:
results = []

for i, item in enumerate(data, start=1):
    img_path = f"vqa_mini/images/COCO_val2014_{item['image_id']:012d}.jpg"
    image = Image.open(img_path).convert("RGB")

    pred = vqa(image=image, question=item["question"])[0]["answer"]

    results.append({
        "question": item["question"],
        "prediction": pred,
        "ground_truth": item["answers"]
    })

    print(f"[{i}/{len(data)}]")
    print("Q:", item["question"])
    print("Pred:", pred)
    print("-" * 40)

with open("vqa_outputs.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved predictions")


Passing `generation_config` together with generation-related arguments=({'eos_token_id', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1/20]
Q: Is it daytime?
Pred: no
----------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/20]
Q: Is there another bus behind this one?
Pred: no
----------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[18/20]
Q: What is the man making?
Pred: ties
----------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[19/20]
Q: What does the yellow sign say?
Pred: pedestrian crossing
----------------------------------------


Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[20/20]
Q: What color are the stripes?
Pred: white
----------------------------------------
Saved predictions


In [ ]:
def vqa_accuracy(pred, answers):
    pred = pred.lower().strip()
    matches = sum(pred == a.lower().strip() for a in answers)
    return min(matches / 3.0, 1.0)

scores = [vqa_accuracy(x["prediction"], x["ground_truth"])
          for x in results]

accuracy = sum(scores) / len(scores)
error_rate = 1 - accuracy

print("Samples:", len(scores))
print("Accuracy:", round(accuracy, 3))
print("Error rate (proxy hallucination):", round(error_rate, 3))


Samples: 20
Accuracy: 0.833
Error rate (proxy hallucination): 0.167


In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "Model": "BLIP-VQA",
    "Dataset": "VQA-v2 (val2014 subset)",
    "Samples": len(scores),
    "Accuracy": round(accuracy, 3),
    "Error_rate_proxy": round(error_rate, 3)
}])

df


,Model,Dataset,Samples,Accuracy,Error_rate_proxy
0,BLIP-VQA,VQA-v2 (val2014 subset),20,0.833,0.167


In [ ]:
df.to_csv("results_table.csv", index=False)
print("Saved results_table.csv")


Saved results_table.csv


In [ ]:
from google.colab import files
files.download("results_table.csv")
files.download("vqa_subset.json")
files.download("vqa_outputs.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import json

def vqa_accuracy(pred, answers):
    # VQA official-style scoring: min(#humans that gave pred / 3, 1)
    pred = str(pred).strip().lower()
    matches = sum(pred == str(a).strip().lower() for a in answers)
    return min(matches / 3.0, 1.0)

with open("vqa_outputs.json") as f:
    outputs = json.load(f)

scores = [vqa_accuracy(x["prediction"], x["ground_truth"]) for x in outputs]

acc = sum(scores) / len(scores)
err = 1 - acc

print("Samples:", len(scores))
print("VQA Accuracy:", round(acc, 3))
print("Error Rate:", round(err, 3))


Samples: 20
VQA Accuracy: 0.833
Error Rate: 0.167


In [ ]:
import pandas as pd

df = pd.DataFrame([{
    "Model": "Salesforce/blip-vqa-base",
    "Dataset": "VQA-v2 val2014 (subset)",
    "N": len(scores),
    "Accuracy (VQA)": round(acc, 3),
    "Error Rate": round(err, 3),
}])

df


,Model,Dataset,N,Accuracy (VQA),Error Rate
0,Salesforce/blip-vqa-base,VQA-v2 val2014 (subset),20,0.833,0.167


In [ ]:
from google.colab import files
files.download("results_table.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd
import json

with open("vqa_outputs.json") as f:
    outputs = json.load(f)

lab_df = pd.DataFrame([{
    "question": x["question"],
    "prediction": x["prediction"],
    "ground_truth_examples": "|".join(x["ground_truth"][:5]),
    "is_hallucinated (0/1)": ""   # you fill this manually
} for x in outputs])

lab_df.to_csv("hallucination_labels.csv", index=False)
print("Saved hallucination_labels.csv (fill is_hallucinated column manually)")



Saved hallucination_labels.csv (fill is_hallucinated column manually)


In [ ]:
from google.colab import files
files.download("hallucination_labels.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df_lab = pd.read_csv("hallucination_labels.csv")
hall_rate = df_lab["is_hallucinated (0/1)"].astype(int).mean()

print("Hallucination rate:", round(hall_rate, 3))


FileNotFoundError: [Errno 2] No such file or directory: 'hallucination_labels.csv'

In [1]:
import json

in_path = "AdvanceTopicPaper.ipynb"
out_path = "AdvanceTopicPaper_clean.ipynb"

with open(in_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widgets metadata if present
nb_meta = nb.get("metadata", {})
if "widgets" in nb_meta:
    nb_meta.pop("widgets", None)
    nb["metadata"] = nb_meta

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Saved:", out_path)

FileNotFoundError: [Errno 2] No such file or directory: 'AdvanceTopicPaper.ipynb'

In [4]:
import os

base = "/content/drive/MyDrive"
for root, dirs, files in os.walk(base):
    for f in files:
        if f.endswith(".ipynb"):
            print(os.path.join(root, f))

/content/drive/MyDrive/Colab Notebooks/session10 (7).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_10 (4).ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/Session_3 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_5.ipynb
/content/drive/MyDrive/Colab Notebooks/Session_6 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_7 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_9 (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_10 (3).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_9_solutions (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_10 (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_11 (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_11_solutions (2).ipynb
/content/drive/MyDrive/Colab Notebooks/Session_12 (

In [ ]:
import json

in_path = "/content/drive/MyDrive/Colab Notebooks/AdvanceTopicPaper.ipynb"
out_path = "/content/AdvanceTopicPaper_clean.ipynb"

with open(in_path, "r", encoding="utf-8") as f:
    nb = json.load(f)

# Remove broken widget metadata
nb.get("metadata", {}).pop("widgets", None)

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("Clean notebook saved")